In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Necessary Libraries


In [26]:
from sklearn.datasets import load_breast_cancer

In [38]:
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV
from sklearn.preprocessing import StandardScaler,RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.pipeline import Pipeline
import joblib 


from sklearn.feature_selection import SelectKBest,f_classif



# Data Loading and Data Selection

In [27]:
data=load_breast_cancer()
df=pd.DataFrame(data.data, columns=data.feature_names)
y=data.target
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# Feature Engineering and Pipeline 

In [30]:
df['radius_texture_ratio']=df['mean radius']/df['mean texture']

X_train,X_test,y_train,y_test=train_test_split(df,y,test_size=0.2,random_state=42)
pipeline=Pipeline([
    ('scaler',StandardScaler()),('selector',SelectKBest(score_func=f_classif,k=10)),
    ('clf',RandomForestClassifier(random_state=42))
])


# > **HyperParameter Tuning**# 


In [2]:
param_grid={
    'clf__n_estimators':[100,200],
    'clf__max_depth':[None,10,20],
    'selector__k':[5,10,15]  #Tuning how many features to keep
}


grid_search = GridSearchCV(pipeline,param_grid,cv=5,scoring='accuracy',n_jobs=1)
grid_search.fit(X_train,y_train)

NameError: name 'GridSearchCV' is not defined

# Evaluation and cross validation


In [33]:
best_model = grid_search.best_estimator_
cv_scores=cross_val_score(best_model,X_train,y_train,cv=5)
print(f'Best Params: {grid_search.best_params_}')
print(f"CV accuracy: {np.mean(cv_scores):.4f} ")

Best Params: {'clf__max_depth': None, 'clf__n_estimators': 100, 'selector__k': 10}
CV accuracy: 0.9473 


# Final Testing


In [39]:
y_pred=best_model.predict(X_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[40  3]
 [ 2 69]]
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [40]:
joblib.dump(best_model,'ml_workflow_model.pkl')
print("Model saved for deployment")

Model saved for deployment
